In [ ]:
from ChatBotLLM import RunLLM 
import os
import json 
import pandas as pd 
import ast

# Single-run models

This workflow allows you to make simple queries to a local LLM with text outputs. Before you run this chunk, ensure that you:
1. Downloaded a model in LM studio (Gemma 3 4B or Deepseek R1 0528 Qwen 3 8B worked well in my case)
2. Loaded the model in LM studio
3. Pre-filled the configuration file model_config.txt in the ChatBotLLM folder


In [ ]:
prompt = "Hello! What is the capital of Russia?"
model_out = RunLLM.single_run(prompt)
print(model_out) 

# Tokeniser

Tokenisers can come in handy when working with RAG pipelines. They transform a text intput into a a vector of numbers. Before you run this chunk, ensure that you:
1. Downloaded and activated a tokeniser in LM studio (I recommend text-embedding-bge-large-en-v1.5)
2. Loaded the model in LM studio
3. Pre-filled the configuration file embedding_model_config.txt in the ChatBotLLM folder


In [ ]:
string_to_tokenise = "Hello! Please tokenise me"
tokenised_string = RunLLM.tokeniser_single_run(string_to_tokenise)
print(tokenised_string[:10]) # Print the first 10 elements of the vector 

# Multimodal workflow

Before you run the script, ensure that you:

1. Downloaded and activated a multimodal-compatible model in LM studio (I recommend qwen3-vl-8b)
2. Loaded the model in LM studio
3. Pre-filled the configuration file multimodal_model_config.txt in the ChatBotLLM folder

In [ ]:
folder_path = "path_to_your_folder_with_images" 
list_of_photos = os.listdir(folder_path)


# Let's order the model to return structured output
def make_prompt(file_name):
   prompt =  f'''
   What do you see in this picture? Please return a strictly formatted json in the format:
   {{file_name: description}}

   The file name is {file_name}
   Return only the requested json without additional commentry, as the output will be passed automatically further down my script.
    '''
   return prompt 



In [ ]:
output_list = []
out_df = pd.DataFrame()
n = len(list_of_photos)
count = 0
for f in list_of_photos:
    prompt = make_prompt(f)
    file_path = fr"{folder_path}\{f}"
    out = RunLLM.single_run_multimodal(file_path, prompt)
    out_dict = ast.literal_eval(out)
    output_list.append(out)
    df_temp = pd.DataFrame.from_dict([out_dict])
    out_df = pd.concat([out_df, df_temp])
    count+=1
    print(f"Done with {count}/{n}")
    # Save progress on the go 
    out_df.to_csv(fr"{folder_path}\Saved file.csv", index=False)



In [ ]:
#Print the output
out_df